# KKBOX Music Recommendation — Data Overview

## Project goal

Predict the probability that a user listens to a song again within
one month of their first observed listening event for that song.

## This notebook

We will inspect the dataset structure, understand the relationships
between the files, and check data quality before building a model.

## Initial observations

- The dataset contains six CSV files.
- The training data includes user IDs, song IDs, listening context,
  and a binary target.
- The target is 1 when repeat listening occurs within the defined
  window, and 0 otherwise.
- The original CSV files are kept in `data/raw` and excluded from Git.

In [1]:
from pathlib import Path

# Notebooks may start in the project folder or the notebooks folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data folder not found: {DATA_DIR}")

files = sorted(DATA_DIR.glob("*.csv"))

print(f"Data folder: {DATA_DIR}")
print(f"CSV files found: {len(files)}\n")

for file in files:
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name}: {size_mb:,.1f} MB")

Data folder: /Users/mariutzadanilescu/Documents/Coding projects/kkbox-music-recommendation/data/raw
CSV files found: 6

members.csv: 2.4 MB
sample_submission.csv: 28.2 MB
song_extra_info.csv: 172.6 MB
songs.csv: 211.6 MB
test.csv: 331.7 MB
train.csv: 926.7 MB


In [2]:
import pandas as pd

# Read only five rows while we inspect the structure.
train_preview = pd.read_csv(DATA_DIR / "train.csv", nrows=5)

print("Training columns:")
print(train_preview.columns.tolist())

display(train_preview)

Training columns:
['msno', 'song_id', 'source_system_tab', 'source_screen_name', 'source_type', 'target']


,msno,song_id,source_system_tab,source_screen_name,source_type,target
0,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,BBzumQNXUHKdEBOB7mAJuzok+IJA1c2Ryg/yzTF6tik=,explore,Explore,online-playlist,1
1,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,bhp/MpSNoqoxOIB+/l8WPqu6jldth4DIpCm3ayXnJqM=,my library,Local playlist more,local-playlist,1
2,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,JNWfrrC7zNN7BdMpsISKa4Mw+xVJYNnxXh3/Epw7QgY=,my library,Local playlist more,local-playlist,1
3,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,2A87tzfnJTSWqD7gIZHisolhe4DMdzkbd6LzO1KHjNs=,my library,Local playlist more,local-playlist,1
4,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,3qm6XTZ6MOCU11x8FIVbAGH5l5uMkT3/ZalWG1oo2Gc=,explore,Explore,online-playlist,1


In [3]:
# Read only the target column, in manageable chunks.
target_counts = pd.Series(0, index=[0, 1], dtype="int64")

for chunk in pd.read_csv(
    DATA_DIR / "train.csv",
    usecols=["target"],
    dtype={"target": "int8"},
    chunksize=250_000,
):
    counts = chunk["target"].value_counts()
    target_counts = target_counts.add(counts, fill_value=0)

target_summary = pd.DataFrame({
    "count": target_counts.astype("int64"),
    "percentage": (target_counts / target_counts.sum() * 100).round(2),
})
target_summary.index.name = "target"

print(f"Total training rows: {int(target_counts.sum()):,}")
display(target_summary)

Total training rows: 7,377,418


,count,percentage
target,,
0,3662762,49.65
1,3714656,50.35


## Target distribution

Approximately 50.35% of training examples have target 1
(repeat listening), and 49.65% have target 0.

The target classes are nearly balanced. This describes how common
each outcome is; it does not yet tell us whether a model can
distinguish between them.

These percentages were calculated from the full training target
column, rather than inferred from the five-row preview.

In [4]:
songs_preview = pd.read_csv(DATA_DIR / "songs.csv", nrows=5)

display(songs_preview)

,song_id,song_length,genre_ids,artist_name,composer,lyricist,language
0,CXoTN1eb7AI+DntdU1vbcwGRV4SCIDxZu+YD8JP8r4E=,247640,465,張信哲 (Jeff Chang),董貞,何啟弘,3.0
1,o0kFgae9QtnYgRkVPqLJwa05zIhRlUjfF7O1tDw0ZDU=,197328,444,BLACKPINK,TEDDY| FUTURE BOUNCE| Bekuh BOOM,TEDDY,31.0
2,DwVvVurfpuz+XPuFvucclVQEyPqcpUkHR0ne1RQzPs0=,231781,465,SUPER JUNIOR,NaN,NaN,31.0
3,dKMBWoZyScdxSkihKG+Vf47nc18N9q4m58+b4e7dSSE=,273554,465,S.H.E,湯小康,徐世珍,3.0
4,W3bqWd3T+VeHFzHAUfARgW9AvVRaF4N5Yzm4Mr6Eo/o=,140329,726,貴族精選,Traditional,Traditional,52.0


In [5]:
members_preview = pd.read_csv(DATA_DIR / "members.csv", nrows=5)

display(members_preview)

,msno,city,bd,gender,registered_via,registration_init_time,expiration_date
0,XQxgAYj3klVKjR3oxPPXYYFp4soD4TuBghkhMTD4oTw=,1,0,NaN,7,20110820,20170920
1,UizsfmJb9mV54qE9hCYyU07Va97c0lCRLEQX3ae+ztM=,1,0,NaN,7,20150628,20170622
2,D8nEhsIOBSoE6VthTaqDX8U6lqjJ7dLdr72mOyLya2A=,1,0,NaN,4,20160411,20170712
3,mCuD+tZ1hERA/o5GPqk38e041J8ZsBaLcu7nGoIIvhI=,1,0,NaN,9,20150906,20150907
4,q4HRBfVSssAFS9iRfxWrohxuk9kCYMKjHOEagUMV6rQ=,1,0,NaN,4,20170126,20170613


In [6]:
# Load all user records.
members = pd.read_csv(DATA_DIR / "members.csv")

print(f"Users table: {members.shape[0]:,} rows")
print(f"Columns: {members.shape[1]}")

# Calculate the percentage of missing values in each column.
missing_percent = members.isna().mean().mul(100).round(2)

display(missing_percent.to_frame(name="missing_percent"))

Users table: 34,403 rows
Columns: 7


,missing_percent
msno,0.00
city,0.00
bd,0.00
gender,57.85
registered_via,0.00
registration_init_time,0.00
expiration_date,0.00


In [7]:
# Check whether each user has exactly one record.
duplicate_ids = members["msno"].duplicated().sum()
print(f"Duplicate user IDs: {duplicate_ids:,}")

# Inspect the recorded ages without changing them.
ages = members["bd"]

print("\nAge summary:")
display(ages.describe().to_frame(name="recorded_age"))

print(f"Age equal to zero: {(ages == 0).sum():,}")
print(f"Negative age: {(ages < 0).sum():,}")
print(f"Age above 100: {(ages > 100).sum():,}")

Duplicate user IDs: 0

Age summary:


,recorded_age
count,34403.000000
mean,12.280935
std,18.170251
min,-43.000000
25%,0.000000
50%,0.000000
75%,25.000000
max,1051.000000


Age equal to zero: 19,932
Negative age: 2
Age above 100: 22


In [8]:
# Preserve bd and create a separate age column.
members["age"] = members["bd"].where(
    members["bd"].between(1, 100)
)

print(f"Missing cleaned age: {members['age'].isna().mean():.2%}")

display(members[["bd", "age"]].head(10))

Missing cleaned age: 58.01%


,bd,age
0,0,NaN
1,0,NaN
2,0,NaN
3,0,NaN
4,0,NaN
5,43,43.0
6,0,NaN
7,0,NaN
8,0,NaN
9,0,NaN


## User metadata findings

- The members table contains 34,403 records with unique user IDs.
- Gender is missing for 57.85% of users.
- Recorded ages include 19,932 zeros, 2 negative values,
  and 22 values above 100.

### Initial age-cleaning rule

We preserve the original `bd` column and create an `age` column.
Ages outside 1–100 are marked as missing, leaving 58.01% missing.

This range is a working assumption, not a guarantee that the
remaining ages are accurate. No user records are removed.